# Analysis 1 — Profiling TFU users inside the Gifter and Follow-Bettor pools

**Question:** within each month, of all Gifters (any gift) and all Follow Bettors (any follow-bet),
how many are TFU, and what do those TFU users look like?

- **Gifter pool** = `total_tip_count + total_box_count + total_wheel_count > 0`
- **Follow Bettor pool** = `total_follow_bet_count > 0`
- **TFU** = sits in **both** pools (gift ✓ AND follow-bet ✓), i.e. the intersection

All analysis is **single-month** — no next-month join, no conversion rate.

Sections:
1. Population overview — pool sizes, TFU count and share within each pool, 6-month trend
2. Segment distribution of TFU users (categorical: account age, watch bucket, device, etc.)
3. Behavioral profile of TFU users (numeric: watch, chat, gifting, betting, breadth)
4. Correlation heatmaps — numeric features vs `is_tfu`, within each pool
5. Monthly trends — how TFU share within each pool moves over 6 months

## 0. Setup

In [ ]:
# --- Colab auth + BigQuery client ---
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', context='notebook')

# TODO: fill in your project / dataset
PROJECT_ID = 'nf-bifrost'
DATASET    = 'your_dataset'                       # <-- update
TABLE      = f'{PROJECT_ID}.{DATASET}.tfu_user_monthly'

client = bigquery.Client(project=PROJECT_ID)

## 1. Pull `tfu_user_monthly` and define pools

Schema columns:
- **IDs/time:** `cust_id`, `data_month`, `month_year`, `month_end`
- **Account:** `account_age_days`, `account_age_tier`, `site`, `currency`
- **Loyalty:** `sessions_count`, `distinct_streamers`, `sessions_bucket`
- **Watch:** `total_watch_sec`, `avg_watch_sec_per_session`, `watch_bucket`
- **Chat:** `total_messages`, `chat_sessions`, `total_bullet_sec`, `total_chatroom_sec`
- **Gifting:** `total_tip_count/usd`, `total_box_count/usd`, `total_wheel_count/usd`
- **Betting:** `total_bet_count`, `total_member_to`, `total_bdw_bet_count`, `total_follow_bet_count`
- **Preferences:** `stream_type_pref`, `device_pref`, `time_segment`, `day_segment`, `breadth_score`
- **Streamer affinity:** `top_follow_streamer*`, `top_gift_streamer*`, `top_streamer_is_same`
- **Target:** `is_tfu`

Pools are **broad/overlapping**: TFU users belong to **both** the Gifter and Follow Bettor pools.

In [ ]:
sql = f"""
SELECT *
FROM `{TABLE}`
"""
df = client.query(sql).to_dataframe()
df['data_month'] = pd.to_datetime(df['data_month'])
print('rows:', len(df), ' users:', df.cust_id.nunique(),
      ' months:', sorted(df.data_month.dt.strftime('%Y-%m').unique()))

In [ ]:
# --- Pool flags (broad / overlapping) ---
df['has_gift'] = ((df['total_tip_count'].fillna(0)
                   + df['total_box_count'].fillna(0)
                   + df['total_wheel_count'].fillna(0)) > 0).astype(int)

df['has_follow_bet'] = (df['total_follow_bet_count'].fillna(0) > 0).astype(int)

# Convenience numeric: total gift USD across tip/box/wheel
df['total_gift_usd'] = (df['total_tip_usd'].fillna(0)
                        + df['total_box_usd'].fillna(0)
                        + df['total_wheel_usd'].fillna(0))

# Sanity check vs is_tfu in the table
df['is_tfu_check'] = ((df['has_gift'] == 1) & (df['has_follow_bet'] == 1)).astype(int)
mismatch = (df['is_tfu'] != df['is_tfu_check']).sum()
print(f'is_tfu sanity mismatches (should be 0): {mismatch}')

# Convenience subframes
gifter  = df[df['has_gift']       == 1].copy()
fbettor = df[df['has_follow_bet'] == 1].copy()
tfu     = df[df['is_tfu']         == 1].copy()

print(f'Gifter pool rows : {len(gifter):,}')
print(f'Follow Bettor pool rows : {len(fbettor):,}')
print(f'TFU rows (intersection) : {len(tfu):,}')

In [ ]:
# === Per-user view (one row per cust_id) ============================
# Loaded directly from `tfu_user_lifetime` — a pre-built BQ table where
# lifetime re-derivation (Pattern A) is applied in SQL:
#   - Numeric features  : SUM across the 6-month window
#   - Categorical labels: re-bucketed on lifetime sums (sessions_bucket,
#     watch_bucket, stream_type_pref, device_pref, time_segment, day_segment)
#   - Breadth score     : distinct activities user EVER did
#   - Top streamers     : re-ranked on lifetime follow-bet / gift USD
#   - is_tfu            : ever-TFU (gift AND follow_bet in the SAME month)
#   - account_age_tier  : computed at the window's latest month_end
# Build SQL: sql/build_tfu_user_lifetime.sql
#
# This replaces the slow in-Python groupby+mode aggregation. Read is fast
# (~1 BQ query) and the bucket logic is consistent with the monthly table.

LIFETIME_TABLE = f'{PROJECT_ID}.{DATASET}.tfu_user_lifetime'

users = client.query(f'SELECT * FROM `{LIFETIME_TABLE}`').to_dataframe()

# Align column names with notebook conventions used in later cells
users['has_gift']       = users['is_gifter']
users['has_follow_bet'] = users['is_follow_bet']

# Convenience total used by some plots
users['total_gift_usd'] = (users['total_tip_usd'].fillna(0)
                           + users['total_box_usd'].fillna(0)
                           + users['total_wheel_usd'].fillna(0))

# Pool sub-frames (ever-in-segment)
gifter_u  = users[users['has_gift']       == 1].copy()
fbettor_u = users[users['has_follow_bet'] == 1].copy()
tfu_u     = users[users['is_tfu']         == 1].copy()

print(f'Unique users (lifetime)  : {len(users):,}')
print(f'  Gifter pool (ever)     : {len(gifter_u):,}')
print(f'  Follow Bettor pool     : {len(fbettor_u):,}')
print(f'  TFU (ever)             : {len(tfu_u):,}')

# Sanity: row count should match df.cust_id.nunique() from cell 5
assert len(users) == df['cust_id'].nunique(), (
    f"Lifetime user count ({len(users):,}) != monthly unique users "
    f"({df['cust_id'].nunique():,}) — check active-user filter parity."
)


## Section 1 — Population overview

Per month:
- Gifter pool size, Follow Bettor pool size, TFU count
- TFU share **within the Gifter pool** = TFU / Gifters
- TFU share **within the Follow Bettor pool** = TFU / Follow Bettors

In [ ]:
overview = (df.groupby('data_month')
              .agg(gifter_pool    = ('has_gift', 'sum'),
                   fbettor_pool   = ('has_follow_bet', 'sum'),
                   tfu_users      = ('is_tfu', 'sum'))
              .reset_index())
overview['tfu_share_of_gifters']  = overview['tfu_users'] / overview['gifter_pool']
overview['tfu_share_of_fbettors'] = overview['tfu_users'] / overview['fbettor_pool']
overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

size_long = overview.melt(id_vars='data_month',
                          value_vars=['gifter_pool', 'fbettor_pool', 'tfu_users'],
                          var_name='pool', value_name='users')
sns.barplot(data=size_long, x='data_month', y='users', hue='pool', ax=axes[0])
axes[0].set_title('Pool sizes per month')
axes[0].tick_params(axis='x', rotation=45)

share_long = overview.melt(id_vars='data_month',
                           value_vars=['tfu_share_of_gifters', 'tfu_share_of_fbettors'],
                           var_name='pool', value_name='tfu_share')
sns.lineplot(data=share_long, x='data_month', y='tfu_share', hue='pool',
             marker='o', linewidth=2, ax=axes[1])
axes[1].set_title('TFU share within each pool')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

## Section 2 — Segment distribution of TFU users

For every categorical column, within **each pool** (Gifter pool, Follow Bettor pool), produce a table with one row per segment value:

| column | description |
|---|---|
| `user_count` | number of users in this segment of this pool |
| `tfu_count` | of those, how many are TFU |
| `tfu_rate` | `tfu_count / user_count` — TFU prevalence within the segment |
| `tfu_share` | `tfu_count / total TFU in pool` — segment's share of the pool's TFU population |

`tfu_rate` answers "how dense is TFU in this segment?" — `tfu_share` answers "where does our TFU population come from?". A segment can have high rate but tiny share (small, dense pocket) or low rate but big share (large, diffuse source).

In [ ]:
# Group label — at user level (ever-in-segment).
# A user can carry multiple labels; here we pick the most informative one:
#   TFU              : was TFU in ANY month (gift + follow-bet in same month)
#   Gifter, non-TFU  : ever gifted but never TFU
#   Follow Bettor,
#       non-TFU      : ever follow-bet but never TFU
#   Other            : never any of the above
users['group'] = np.select(
    [users['is_tfu'] == 1,
     (users['has_gift']        == 1) & (users['is_tfu'] == 0),
     (users['has_follow_bet']  == 1) & (users['is_tfu'] == 0)],
    ['TFU', 'Gifter, non-TFU', 'Follow Bettor, non-TFU'],
    default='Other'
)

group_order = ['TFU', 'Gifter, non-TFU', 'Follow Bettor, non-TFU']
print('Group distribution — UNIQUE USERS:')
print(users['group'].value_counts())


In [ ]:
categorical_cols = [
    'account_age_tier', 'watch_bucket', 'sessions_bucket',
    'time_segment', 'day_segment',
    'stream_type_pref', 'device_pref',
    'dominant_league',
]

def segment_table(pool_df, col):
    """One row per segment value. Counts are UNIQUE USERS in `pool_df`."""
    total_tfu = pool_df['is_tfu'].sum()           # ever-TFU users in pool
    t = (pool_df.groupby(col)
                 .agg(user_count=('cust_id', 'nunique'),
                      tfu_count =('is_tfu',  'sum'))
                 .reset_index())
    t['tfu_rate']  = t['tfu_count'] / t['user_count']
    t['tfu_share'] = t['tfu_count'] / total_tfu
    t = t.sort_values('tfu_share', ascending=False).reset_index(drop=True)
    totals = pd.DataFrame({
        col: ['TOTAL'],
        'user_count': [t['user_count'].sum()],
        'tfu_count':  [t['tfu_count'].sum()],
        'tfu_rate':   [t['tfu_count'].sum() / t['user_count'].sum()],
        'tfu_share':  [t['tfu_share'].sum()],
    })
    return pd.concat([t, totals], ignore_index=True)

def show_segment(pool_df, pool_name):
    print(f'\n========== {pool_name}  (unique users={len(pool_df):,},  ever-TFU={int(pool_df["is_tfu"].sum()):,}) ==========')
    for col in categorical_cols:
        print(f'\n--- {col} ---')
        tab = segment_table(pool_df, col)
        fmt = tab.copy()
        fmt['user_count'] = fmt['user_count'].map('{:,}'.format)
        fmt['tfu_count']  = fmt['tfu_count'].map('{:,}'.format)
        fmt['tfu_rate']   = fmt['tfu_rate'].map('{:.2%}'.format)
        fmt['tfu_share']  = fmt['tfu_share'].map('{:.2%}'.format)
        print(fmt.to_string(index=False))

show_segment(gifter_u,  'Gifter pool')
show_segment(fbettor_u, 'Follow Bettor pool')


In [ ]:
# Visualize: tfu_rate (line) and tfu_share (bars) per segment, per pool — UNIQUE USERS
def segment_plot(pool_df, pool_name, col):
    t = segment_table(pool_df, col)
    t = t[t[col] != 'TOTAL']
    fig, ax1 = plt.subplots(figsize=(9, 4))
    sns.barplot(data=t, x=col, y='tfu_share', color='#9ecae1', ax=ax1)
    ax1.set_ylabel('tfu_share (bars)', color='#3182bd')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax1.tick_params(axis='x', rotation=30)
    for label in ax1.get_xticklabels():
        label.set_ha('right')

    ax2 = ax1.twinx()
    ax2.plot(range(len(t)), t['tfu_rate'].values, color='#e6550d', marker='o', linewidth=2)
    ax2.set_ylabel('tfu_rate (line)', color='#e6550d')
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))

    plt.title(f'{pool_name} — {col}  (unique users)')
    plt.tight_layout(); plt.show()

for col in categorical_cols:
    segment_plot(gifter_u,  'Gifter pool',        col)
    segment_plot(fbettor_u, 'Follow Bettor pool', col)


## Section 3 — Behavioral profile of TFU users

Compare TFU vs non-TFU on numeric behavior — watch, chat, gifting depth, betting, breadth. Same three groups as Section 2.

In [ ]:
numeric_features = [
    # Watch
    'total_watch_sec', 'avg_watch_sec_per_session',
    # Chat
    'total_messages', 'chat_sessions', 'total_bullet_sec', 'total_chatroom_sec',
    # Gifting
    'total_tip_count', 'total_box_count', 'total_wheel_count',
    'total_tip_usd', 'total_box_usd', 'total_wheel_usd', 'total_gift_usd',
    # Betting
    'total_bet_count', 'total_member_to', 'total_bdw_bet_count', 'total_follow_bet_count',
    # Breadth / loyalty
    'breadth_score', 'sessions_count', 'distinct_streamers',
]

In [ ]:
profile = (users[users['group'].isin(group_order)]
             .groupby('group')[numeric_features]
             .agg(['median', 'mean'])
             .round(2))
profile.loc[group_order]


In [ ]:
# Boxplots (log scale) per numeric feature — UNIQUE USERS
plot_df = users[users['group'].isin(group_order)].copy()

n = len(numeric_features)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    ax = axes[i]
    data = plot_df[[feat, 'group']].copy()
    data[feat] = data[feat].clip(lower=0) + 1
    sns.boxplot(data=data, x='group', y=feat, order=group_order,
                ax=ax, showfliers=False)
    ax.set_yscale('log')
    ax.set_title(feat)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout(); plt.show()


## Section 4 — Correlation heatmaps

Within each pool (Gifter, Follow Bettor), Spearman correlation between numeric features and `is_tfu`. Highlights which behaviors are most associated with being TFU **inside that pool**.

In [ ]:
def corr_heatmap(frame, pool_name):
    cols = numeric_features + ['is_tfu']
    corr = frame[cols].corr(method='spearman')
    plt.figure(figsize=(11, 9))
    sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1,
                annot=False, square=False)
    plt.title(f'Spearman correlation — {pool_name} (unique users n={len(frame):,})')
    plt.tight_layout(); plt.show()

    top = (corr['is_tfu'].drop('is_tfu')
                          .abs().sort_values(ascending=False)
                          .head(15))
    print(f'\nTop |corr| with is_tfu — {pool_name}:')
    print(top.round(3))

corr_heatmap(gifter_u,  'Gifter pool')
corr_heatmap(fbettor_u, 'Follow Bettor pool')


### Section 4b — Leakage-aware correlation (clean feature set)

The first heatmaps include features that are **definitionally tied** to `is_tfu`:

- **Gifter pool**: `total_follow_bet_count` *is* the target (corr ≈ 1.0).
- **Follow Bettor pool**: `total_tip/box/wheel_count`, `*_usd`, `total_gift_usd` *are* the target.
- **Both**: `breadth_score` counts gift + bet activities, so it inherits the leakage.

Re-run the heatmaps after removing the tainted features so we see the **real** behavioral signal.

In [ ]:
# Clean feature lists (per pool) -- drop features that are definitionally tied to is_tfu.
# Also drop breadth_score since it aggregates gift + bet activities.

gift_features = ['total_tip_count', 'total_box_count', 'total_wheel_count',
                 'total_tip_usd',   'total_box_usd',   'total_wheel_usd',
                 'total_gift_usd']
follow_bet_features = ['total_follow_bet_count']
tainted_both = ['breadth_score']

clean_gifter  = [f for f in numeric_features
                 if f not in follow_bet_features + tainted_both]
clean_fbettor = [f for f in numeric_features
                 if f not in gift_features + tainted_both]

print('Gifter-pool clean features  :', clean_gifter)
print('FBettor-pool clean features :', clean_fbettor)

In [ ]:
def corr_heatmap_clean(frame, pool_name, feats):
    cols = feats + ['is_tfu']
    corr = frame[cols].corr(method='spearman')
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1,
                annot=False, square=False)
    plt.title(f'Spearman correlation (clean) — {pool_name} (unique users n={len(frame):,})')
    plt.tight_layout(); plt.show()

    top = (corr['is_tfu'].drop('is_tfu')
                          .abs().sort_values(ascending=False)
                          .head(15))
    print(f'\nTop |corr| with is_tfu (clean) — {pool_name}:')
    print(top.round(3))

corr_heatmap_clean(gifter_u,  'Gifter pool',        clean_gifter)
corr_heatmap_clean(fbettor_u, 'Follow Bettor pool', clean_fbettor)


## Section 5 — Monthly trends

How TFU users move over 6 months: absolute count, share within each pool, and any drift in their categorical mix.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.lineplot(data=overview, x='data_month', y='tfu_users',
             marker='o', linewidth=2, ax=axes[0])
axes[0].set_title('TFU user count — 6-month trend')
axes[0].tick_params(axis='x', rotation=45)

sns.lineplot(data=share_long, x='data_month', y='tfu_share', hue='pool',
             marker='o', linewidth=2, ax=axes[1])
axes[1].set_title('TFU share within each pool — 6-month trend')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

In [ ]:
# Categorical mix drift: TFU-only share of each category per month
for col in categorical_cols:
    t = (tfu.groupby(['data_month', col]).size()
             .groupby(level=0).apply(lambda s: s / s.sum())
             .rename('share').reset_index())
    plt.figure(figsize=(9, 3.5))
    sns.lineplot(data=t, x='data_month', y='share', hue=col, marker='o')
    plt.title(f'TFU mix over time — {col}')
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    plt.xticks(rotation=45)
    plt.tight_layout(); plt.show()

## Takeaways scratch-pad

Fill in after running:
- Avg TFU share of Gifter pool ≈ _ ;  of Follow Bettor pool ≈ _
- TFU users skew toward which `account_age_tier`? _
- TFU users skew toward which `watch_bucket`? _
- Device / time / day-segment skew: _
- Biggest numeric gaps (TFU vs non-TFU): _
- Top features correlated with `is_tfu` in Gifter pool: _
- Top features correlated with `is_tfu` in Follow Bettor pool: _
- 6-month trend: TFU share rising / flat / falling: _